In [ ]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


# ----------------------------------------
# GAP STATISTIC
# ----------------------------------------

def gap_statistic(X, k_max=10, B=20):
    gaps = []
    k_range = range(1, k_max + 1)

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=30).fit(X)
        disp = km.inertia_

        ref = []
        for _ in range(B):
            rand = np.random.uniform(X.min(), X.max(), X.shape)
            ref_km = KMeans(n_clusters=k, random_state=42, n_init=30).fit(rand)
            ref.append(ref_km.inertia_)

        gap = np.mean(np.log(ref)) - np.log(disp)
        gaps.append(gap)

    optimal_k = k_range[np.argmax(gaps)]

    return optimal_k, gaps, list(k_range)


X_model = X_resid.dropna()
optimal_k, gaps, gap_k = gap_statistic(X_model)



# ----------------------------------------

k_range = range(1, 11)

calinski_scores = []
silhouette_scores = []
inertia_scores = []
davies_bouldin_scores = []

for k_eval in k_range:
    kmeans = KMeans(n_clusters=k_eval, random_state=42, n_init=30)
    labels = kmeans.fit_predict(X_model)

    calinski_scores.append(
        calinski_harabasz_score(X_model, labels)
    )

    silhouette_scores.append(
        silhouette_score(X_model, labels)
    )

    inertia_scores.append(
        kmeans.inertia_
    )

    davies_bouldin_scores.append(
        davies_bouldin_score(X_model, labels)
    )



# ----------------------------------------

# important: set k either manually after decision based on all cluster metrices (recommended) 
# or automatically based on GAP-statistic (not recommended, evaluate all metrices for best cluster solution)
manual_k = 2

if manual_k is None:
    k = optimal_k
    print(f"Verwende optimales k aus Gap Statistic: {k}")
else:
    k = manual_k
    print(f"Verwende manuell gesetztes k: {k}")

final_kmeans = KMeans(n_clusters=k, random_state=42, n_init=30)
clusters = final_kmeans.fit_predict(X_model)

df_final = df_clean.loc[X_model.index].copy()
df_final['Cluster'] = clusters + 1



# ----------------------------------------
# Table of all Cluster Metrices
# ----------------------------------------

metrics_table = pd.DataFrame({
    "k": list(k_range),
    "Calinski_Harabasz": calinski_scores,
    "Silhouette": silhouette_scores,
    "Inertia": inertia_scores,
    "Davies_Bouldin": davies_bouldin_scores
})

metrics_table.to_excel(
    os.path.join(folder_path, f"Cluster_Metriken_k{k}.xlsx"),
    index=False
)

print("\nCluster Validation Metrics (k = 2 bis 10):")
print(metrics_table.round(3))

# ----------------------------------------
# CLUSTER × DIAGNOSIS
# ----------------------------------------

ct = pd.crosstab(df_final['Cluster'], df_final['Group'])
print("\nCluster × Group:")
print(ct)

print("\nCluster sizes:")
print(df_final['Cluster'].value_counts())